I used this model to test out building and training the model. I used colab's GPUs. 

In [ ]:
%pip install transformers h5py pandas pyarrow protobuf

In [3]:
# %pip install torch

# install cuda on windows
%pip install torch --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126
   ---------------------------------------- 0.0/2.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 GB 21.8 MB/s eta 0:01:59
   ---------------------------------------- 0.0/2.6 GB 25.9 MB/s eta 0:01:40
   ---------------------------------------- 0.0/2.6 GB 26.3 MB/s eta 0:01:38
   ---------------------------------------- 0.0/2.6 GB 27.2 MB/s eta 0:01:35
   ---------------------------------------- 0.0/2.6 GB 28.2 MB/s eta 0:01:31
    --------------------------------------- 0.0/2.6 GB 29.1 MB/s eta 0:01:28
    --------------------------------------- 0.0/2.6 GB 28.1 MB/s eta 0:01:31
    --------------------------------------- 0.0/2.6 GB 27.8 MB/s eta 0:01:32
    --------------------------------------- 0.1/2.6 GB 27.9 MB/s eta 0:01:32
    --------------------------------------- 0.1/2.6 GB 27.3 MB/s eta 0:01:33
    --------------------------------------- 0.1/2.6 GB 27.2 MB/s eta 0:01:34
   - --------------------

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

c:\Users\jackm\miniconda3\envs\capstone-model\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [13]:
from torch.utils.data import DataLoader
from ConvStyleDataset import AudioHDF5Dataset, collate_pad

H5_PATH     = "../data/processed/merged_audio.h5"
META_PATH   = "../data/processed/merged_metadata.parquet"
BATCH_SIZE = 4      # for my baby gpu
SAMPLE_RATE = 16_000

dataset = AudioHDF5Dataset(
    h5_path=H5_PATH,
    meta_path=META_PATH,
    meta_columns=["transcription"],
    sample_rate=SAMPLE_RATE,
    # no max_len_sec -- collate_pad handles variable lengths
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_pad,
    num_workers=0,
)

In [5]:
# data batch test
batch = next(iter(loader))

audio   = batch["audio"]          # (B, T_max) float32, zero-padded
lengths = batch["lengths"]        # (B,) actual frame counts
texts   = batch["transcription"]  # list[str]

print("audio shape  :", audio.shape)
print("audio dtype  :", audio.dtype)
print("lengths      :", lengths)
print("min/max len  :", lengths.min().item(), "/", lengths.max().item())
print("transcription sample:", texts[0])
print()

# verify padding is actually zero beyond each utterance's real length
for i in range(len(lengths)):
    L = lengths[i].item()
    tail = audio[i, L:]
    # skip if this sample is the longest in the batch
    if tail.numel() > 0:
        assert tail.abs().max() == 0, f"item {i}: non-zero values past length {L}"

print("Padding check passed")

audio shape  : torch.Size([8, 2963520])
audio dtype  : torch.float32
lengths      : tensor([ 285120,  102720, 2963520,  415200,  361920,  192480,  120480,  171360])
min/max len  : 102720 / 2963520
transcription sample:  I see the bomb, I see, excuse me.

Padding check passed


In [ ]:
from transformers import BertTokenizer, BertModel, WavLMModel, Wav2Vec2FeatureExtractor

bert_tokenizer  = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model      = BertModel.from_pretrained("bert-base-uncased").half().to(device)  # using half() to limit gpu memory for quick tests

# WavLM only needs a feature extractor, not a full processor with tokenizer
wavlm_processor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
wavlm_model     = WavLMModel.from_pretrained("microsoft/wavlm-base-plus").half().to(device)

for model in (bert_model, wavlm_model):
    for param in model.parameters():
        param.requires_grad = False
    model.eval()

print("Encoders loaded and frozen")
print(f"allocated after loading: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2617.85it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 248/248 [00:00<00:00, 13768.38it/s]


Encoders loaded and frozen


In [8]:
class SelfAttentivePooling(nn.Module):

    def __init__(self, dim: int):
        super().__init__()
        self.scorer = nn.Linear(dim, 1, bias=False)

    def forward(self, hidden: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        # hidden: (B, T, d),  mask: (B, T) -- 1 for real tokens, 0 for padding
        scores = self.scorer(hidden).squeeze(-1)  # (B, T)

        if mask is not None:
            # push padding positions to -inf so softmax zeroes them out
            scores = scores.masked_fill(~mask.bool(), float("-inf"))

        weights = F.softmax(scores, dim=-1)
        return (weights.unsqueeze(-1) * hidden).sum(dim=1)  # (B, d)

In [9]:
class ModalityEncoder(nn.Module):

    def __init__(self, backbone, pooler):
        super().__init__()
        self.backbone = backbone
        self.pooler   = pooler

    def forward(self, hidden: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        return self.pooler(hidden, mask)  # (B, d)

In [10]:
def encode_text_inputs(texts, tokenizer):
    return tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(device)

def encode_audio_inputs(audio, lengths, processor):
    inputs = processor(
        list(audio.cpu().numpy()),
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
        padding=True,
    ).to(device)
    return inputs, lengths

In [ ]:
class DualModalityEmbedder(nn.Module):

    def __init__(self, text_encoder, audio_encoder, tokenizer, processor):
        super().__init__()
        self.text_encoder  = text_encoder
        self.audio_encoder = audio_encoder
        self.tokenizer     = tokenizer
        self.processor     = processor

    def embed_text(self, texts):
        tokens = encode_text_inputs(texts, self.tokenizer)
        with torch.no_grad():
            hidden = self.text_encoder.backbone(**tokens).last_hidden_state
        return self.text_encoder(hidden, tokens["attention_mask"])

    def embed_audio(self, audio, lengths):
        inputs, lengths = encode_audio_inputs(audio, lengths, self.processor)
        with torch.no_grad():
            hidden = self.audio_encoder.backbone(**inputs).last_hidden_state

        # resize the mask to match WavLM's downsampled time dimension
        T_out    = hidden.shape[1]
        raw_mask = (torch.arange(audio.shape[1], device=device).unsqueeze(0)
                    < lengths.unsqueeze(1).to(device)).float()
        mask_ds  = F.interpolate(raw_mask.unsqueeze(1), size=T_out, mode="nearest").squeeze(1).bool()

        return self.audio_encoder(hidden, mask_ds)

    def forward(self, audio, lengths, texts):
        # convenience method, downstream transformers should call embed_text/embed_audio directly
        return self.embed_text(texts), self.embed_audio(audio, lengths)


text_encoder  = ModalityEncoder(bert_model,  SelfAttentivePooling(768)).to(device)
audio_encoder = ModalityEncoder(wavlm_model, SelfAttentivePooling(768)).to(device)

embedder = DualModalityEmbedder(
    text_encoder, audio_encoder, bert_tokenizer, wavlm_processor
).to(device)

print("Embedder ready")

Embedder ready


In [ ]:
embedder.eval()


wavlm_model.to(device)
audio_emb = embedder.embed_audio(batch["audio"].to(device), batch["lengths"])
wavlm_model.to("cpu")
torch.cuda.empty_cache()

bert_model.to(device)
text_emb = embedder.embed_text(batch["transcription"])
bert_model.to("cpu")
torch.cuda.empty_cache()

print("text_emb shape :", text_emb.shape)
print("audio_emb shape:", audio_emb.shape)

assert text_emb.shape  == (BATCH_SIZE, 768)
assert audio_emb.shape == (BATCH_SIZE, 768)
assert not torch.allclose(text_emb, audio_emb), "embeddings are suspiciously identical"

print("All assertions passed")

In [19]:
torch.cuda.empty_cache()
print(f"allocated : {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"reserved  : {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

allocated : 11.04 GB
reserved  : 11.12 GB
